# 04 - NAFNet Training (Google Colab)
Train the NAFNet restoration model on the SEM dataset.

**Before running:** Set `Runtime -> Change runtime type -> GPU (T4)`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/<your-username>/semiconductor-image-restoration.git /content/repo
%cd /content/repo
!pip install -r requirements.txt -q

In [ ]:
import zipfile, os
zip_path = "/content/drive/MyDrive/train.zip"  # update to your uploaded zip path
extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_path)

import glob
noisy_dir = glob.glob(f"{extract_path}/**/NoisyLR", recursive=True)[0]
data_root = os.path.dirname(noisy_dir)
print("data_root:", data_root, "| contents:", os.listdir(data_root))

In [ ]:
# Train directly via the training script (recommended - handles checkpointing)
!python src/train.py \
    --data_dir {data_root} \
    --noisy_folder NoisyLR \
    --clean_folder CleanHR \
    --epochs 50 \
    --batch_size 16 \
    --lr 2e-4 \
    --model_size small \
    --checkpoint_dir /content/drive/MyDrive/nafnet_checkpoints

In [ ]:
# Alternative: interactive training loop (for experimentation / visualization mid-training)
import sys
sys.path.append('src')
import torch
from model import build_model
from preprocessing import train_val_split
from metrics import evaluate_batch
from torch.utils.data import DataLoader
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_ds, val_ds = train_val_split(data_root, val_fraction=0.1)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

model = build_model(in_channels=1, size='small').to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
l1 = nn.L1Loss()
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M params on {device}")

In [ ]:
best_psnr = 0
for epoch in range(50):
    model.train()
    total_loss = 0
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        opt.zero_grad()
        restored = model(noisy)
        loss = l1(restored, clean)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    sched.step()

    model.eval()
    val_psnr = 0
    with torch.no_grad():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            restored = model(noisy).clamp(0,1)
            val_psnr += evaluate_batch(restored, clean)['psnr']
    val_psnr /= len(val_loader)

    print(f"Epoch {epoch+1}/50 | loss={total_loss/len(train_loader):.4f} | val_psnr={val_psnr:.2f}dB")
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save({'model_state': model.state_dict()}, '/content/drive/MyDrive/nafnet_best.pth')
        print("  -> saved best checkpoint")